# Lab 3.10 &mdash; Text Generation: Greedy vs Sampling

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 2 &middot; Module 3 &mdash; Why Transformers?**

### What you'll do
- Build a tiny next-word model and generate from it
- Implement greedy decoding (always the top token)
- Implement temperature-controlled softmax sampling

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

**Reference:** [Module 3 slides &mdash; Encoder vs decoder (BERT vs GPT)](../../presentation/day2-module3-why-transformers.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 3 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = "/tmp/biaa-lab-03-10"
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
GPT-style models **generate** by repeatedly predicting the **next token** and feeding it back in. Two
decoding strategies: **greedy** (always take the most likely token) and **sampling** with a
**temperature** (lower = safer/sharper, higher = more random/creative). We build a tiny word-level
model so the mechanics are crystal clear. The client's GPT-API generation is the same loop at scale &mdash;
the optional cell at the end shows the real thing.

In [ ]:
# DEMO -- build a tiny next-word model from a corpus
from collections import defaultdict, Counter
CORPUS = "the cat sat on the mat the cat sat on the rug the cat ran fast".split()
MODEL = defaultdict(Counter)
for a, b in zip(CORPUS, CORPUS[1:]): MODEL[a][b] += 1
print("after 'the':", dict(MODEL["the"]))

## Your Turn
Implement temperature **softmax**, **greedy** next-token, and the **generate** loop.

In [ ]:
import numpy as np
from collections import defaultdict, Counter
CORPUS = "the cat sat on the mat the cat sat on the rug the cat ran fast".split()
MODEL = defaultdict(Counter)
for a, b in zip(CORPUS, CORPUS[1:]): MODEL[a][b] += 1

def softmax_temp(logits, temp=1.0):
    z = np.array(logits, dtype=float) / temp
    e = ___   # TODO: exp of (z - z.max()) for stability
    return e / e.sum()

def greedy_next(word):
    return ___   # TODO: the most common next word after `word` (Counter.most_common)

def generate(start, steps=4):
    out = [start]
    for _ in range(steps):
        out.append(___)   # TODO: greedy next word after the last token
    return out

try:
    print('greedy:', ' '.join(generate('the', 4)))
    print('temp 0.5 vs 2.0 max prob:', round(softmax_temp([2,1,0],0.5).max(),3), round(softmax_temp([2,1,0],2.0).max(),3))
except Exception as e: print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

import numpy as np
expect_true("softmax_temp sums to 1", lambda: abs(float(softmax_temp([2.0,1.0,0.0]).sum()) - 1.0) < 1e-9)
expect_true("lower temperature is sharper (higher max prob)", lambda: softmax_temp([2,1,0], 0.5).max() > softmax_temp([2,1,0], 2.0).max())
expect_true("greedy_next('the') == 'cat'", lambda: greedy_next("the") == "cat")
expect_true("generate('the', 4) starts the cat sat on", lambda: generate("the", 4)[:4] == ["the", "cat", "sat", "on"])

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; the real thing with Hugging Face (not graded)
Generate text with a real (tiny) GPT-style model. To instead use the OpenAI/Groq API, set your key in the marked spot &mdash; never commit keys. Safe to skip &mdash; it needs `pip install transformers torch` and a one-time model
download. If `transformers` is not installed, the cell simply prints a note and moves on.

In [ ]:
try:
    from transformers import pipeline, set_seed
    set_seed(0)
    gen = pipeline("text-generation", model="sshleifer/tiny-gpt2")
    print(gen("The future of AI is", max_new_tokens=15)[0]["generated_text"])
except Exception as e:
    print("transformers not installed -- optional demo skipped.", type(e).__name__)

# --- OPTIONAL: real GPT API (needs your own key; not graded) ---
# import os; from openai import OpenAI
# client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))   # set OPENAI_API_KEY yourself
# print(client.chat.completions.create(model="gpt-4o-mini",
#       messages=[{"role":"user","content":"Write one sentence about transformers."}]).choices[0].message.content)

---
### Key takeaway
Greedy + temperature are the decoding knobs behind every chat model. You now understand what 'temperature' on the GPT API actually does.

[&#8592; All Module 3 labs](./index.html) &nbsp;&middot;&nbsp; [Module 3 slides](../../presentation/day2-module3-why-transformers.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>